# Transformers 模型量化技术：GPTQ

![image](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/blog/159_autogptq_transformers/thumbnail.jpg)

2022年，Frantar等人发表了论文 [GPTQ：Accurate Post-Training Quantization for Generative Pre-trained Transformers](https://arxiv.org/abs/2210.17323)。

这篇论文详细介绍了一种训练后量化算法，适用于所有通用的预训练 Transformer模型，同时只有微小的性能下降。

GPTQ算法需要通过对量化模型进行推理来校准模型的量化权重。详细的量化算法在[原始论文](https://arxiv.org/pdf/2210.17323.pdf)中有描述。

基于[`auto-gptq`](https://github.com/PanQiWei/AutoGPTQ.git) 开源实现库，transformers 支持使用GPTQ算法量化的模型。

## 使用 GPTQ 量化模型

为了使用 `auto-gptq` 库量化一个模型，我们需要向量化器传递一个数据集。

通常有两种方式构造数据集：
- 量化器支持的默认数据集（包括`['wikitext2','c4','c4-new','ptb','ptb-new']`）
- 一个字符串列表（这些字符串将被用作数据集）

### 使用 GPTQ 算法支持的默认数据集来量化

在下面的示例中，让我们尝试使用`"wikitext2"`数据集将模型量化为4位精度。支持的精度有`[2, 4, 6, 8]`。

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig
import torch

model_name_or_path = "facebook/opt-2.7b"

quantization_config = GPTQConfig(
     bits=4, # 量化精度
     group_size=128,
     dataset="wikitext2",
     desc_act=False,
)

#### 逐层量化

关于 `CUDA extension not installed` 的说明: https://github.com/AutoGPTQ/AutoGPTQ/issues/249

In [2]:
quant_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    quantization_config=quantization_config,
    device_map='auto')

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

CUDA extension not installed.
CUDA extension not installed.
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/5.30G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

... (35 outputs left out)

Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

### 实测 GPTQ 量化模型：GPU显存占用峰值超过10GB

```shell
Wed Feb  7 22:03:42 2024
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.23.08              Driver Version: 545.23.08    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:0D.0 Off |                    0 |
| N/A   66C    P0              69W /  70W |   9719MiB / 15360MiB |    100%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+

+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
|    0   N/A  N/A      2334      C   /root/miniconda3/bin/python               10714MiB |
+---------------------------------------------------------------------------------------+
```

### 检查量化模型正确性

通过检查`线性层的属性`来确保模型已正确量化，它们应该包含`qweight`和`qzeros`属性，这些属性应该是`torch.int32`数据类型。

In [4]:
quant_model.model.decoder.layers[0].self_attn.q_proj.__dict__

{'training': True,
 '_parameters': OrderedDict(),
 '_buffers': OrderedDict([('qweight',
               tensor([[ 1766754698, -1249142373,  1183631034,  ..., -2038658921,
                        -2037544795, -1956877206],
                       [ 1772710025,  1739893370, -1500087466,  ...,  2021033895,
                         -662329995,  1756019066],
                       [ -895658394, -2007414633, -1951893913,  ..., -1429760649,
                         -980833883,  1451914633],
                       ...,
                       [ 2025363833, -1412855115,  1539086490,  ...,  -342189415,
                        -1737062521, -1950833303],
                       [ 2023326856, -1432974698, -1788251003,  ..., -1161734024,
                         2043176297,  1449571741],
                       [ 1769506966, -2021291654, -1182103157,  ...,  1999993942,
                         1753778600,  1970816134]], device='cuda:0', dtype=torch.int32)),
              ('qzeros',
               tensor(

In [12]:
q_proj = quant_model.model.decoder.layers[0].self_attn.q_proj

print("输入特征：", q_proj.infeatures)
print("输出特征：", q_proj.outfeatures)
print("位宽：", q_proj.bits)
print("分组大小：", q_proj.group_size)

print("qweight：", q_proj.qweight.shape, q_proj.qweight.dtype)
print("qzeros：", q_proj.qzeros.shape, q_proj.qzeros.dtype)
print("scales：", q_proj.scales.shape, q_proj.scales.dtype)
print("g_idx：", q_proj.g_idx.shape, q_proj.g_idx.dtype)
print("bias：", q_proj.bias.shape, q_proj.bias.dtype)

print("设备：", q_proj.qweight.device)
print("CUDA 扩展：", q_proj.autogptq_cuda_available)
print("量化层是否可训练：", q_proj.trainable)

输入特征： 2560
输出特征： 2560
位宽： 4
分组大小： 128
qweight： torch.Size([320, 2560]) torch.int32
qzeros： torch.Size([20, 320]) torch.int32
scales： torch.Size([20, 2560]) torch.float16
g_idx： torch.Size([2560]) torch.int32
bias： torch.Size([2560]) torch.float16
设备： cuda:0
CUDA 扩展： False
量化层是否可训练： False


In [5]:
# 保存模型权重
quant_model.save_pretrained("models/opt-2.7b-gptq")

#### 使用 GPU 加载模型并生成文本

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

text = "Merry Christmas! I'm glad to"
inputs = tokenizer(text, return_tensors="pt").to(0)

out = quant_model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(out[0], skip_special_tokens=True))

Merry Christmas! I'm glad to see you're still here.
Thank you! Merry Christmas to you too!


### 使用自定义数据集量化模型（灵活可扩展，前提是准备好数据）

下面演示通过传递自定义数据集来量化一个模型。

通过字符串列表来自定义一个数据集，建议样本数不少于128（样本数太少会影响模型性能）

In [10]:
from transformers import AutoModelForCausalLM, GPTQConfig, AutoTokenizer

model_name_or_path = "facebook/opt-2.7b"
custom_dataset = ["auto-gptq is an easy-to-use model quantization library with user-friendly apis, based on GPTQ algorithm."]

custom_quantization_config = GPTQConfig(
    bits=4,
    group_size=128,
    desc_act=False,
    dataset=custom_dataset
)

custom_quant_model = AutoModelForCausalLM.from_pretrained(model_name_or_path,
                                                          quantization_config=custom_quantization_config,
                                                          torch_dtype=torch.float16,
                                                          device_map="auto")

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Quantizing model.decoder.layers blocks :   0%|          | 0/32 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

... (24 outputs left out)

Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

相比使用默认数据集，未经精心准备的自定义数据集会明显降低模型性能

In [11]:
text = "Merry Christmas! I'm glad to"
inputs = tokenizer(text, return_tensors="pt").to(0)

out = custom_quant_model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(out[0], skip_special_tokens=True))

Merry Christmas! I'm glad to.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.



## [Optional]Homework: 使用 GPTQ 算法量化 OPT-6.7B 模型


# 先用 OPT-125M 冒烟测试

In [1]:
import gc
import os
import time
from pathlib import Path

# 在 CUDA 首次分配前设置，减少显存碎片
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from gptqmodel import GPTQModel, GPTQConfig

/home/sigan/miniconda3/envs/nowLLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


W1101 16:41:57.898000 7197 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1101 16:41:58.009000 7197 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.7.0
Transformers : 5.20.2
Torch        : 2.14.1+cu130
Triton       : 3.8.0


In [2]:
model_id = "facebook/opt-125m"

quant_path = Path(
    "/sigan/ai-ml/models/quantized/"
    "opt-125m-gptqmodel-4bit-smoke"
)

# 防止误覆盖之前的结果
if quant_path.exists():
    raise FileExistsError(
        f"输出目录已存在：{quant_path}\n"
        "请改用新目录，或自行确认后清理旧目录。"
    )

quant_path.mkdir(parents=True, exist_ok=True)

print("模型：", model_id)
print("输出：", quant_path)
print("CUDA：", torch.cuda.is_available())
print("GPU：", torch.cuda.get_device_name(0))

模型： facebook/opt-125m
输出： /sigan/ai-ml/models/quantized/opt-125m-gptqmodel-4bit-smoke
CUDA： True
GPU： NVIDIA GeForce RTX 4080


## 准备校准数据

In [3]:
calibration_dataset = load_dataset(
    "allenai/c4",
    data_files=(
        "en/c4-train.00001-of-01024.json.gz"
    ),
    split="train",
).select(range(16))

print("校准样本数：", len(calibration_dataset))
print(calibration_dataset[0]["text"][:300])

INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/allenai/c4/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/api/resolve-cache/datasets/allenai/c4/1588ec454efa1a09f29cd18ddd04fe05fc8653a2/README.md?%2Fdatasets%2Fallenai%2Fc4%2Fresolve%2Fmain%2FREADME.md=&etag=%222a750bf152e98e6eb2a14a6c2bd58c940c3cf504%22 "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/allenai/c4/resolve/1588ec454efa1a09f29cd18ddd04fe05fc8653a2/c4.py "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/allenai/c4/allenai/c4.py "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/datasets/allenai/c4/revision/1588ec454efa1a09f29cd18ddd04fe05fc8653a2 "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/allenai/c4/resolve/1588ec454efa1a09f29cd18ddd04fe05fc8653a2/.huggingface.yaml "HTTP/1.1 404 No

校准样本数： 16
At Seven Avenue Design we design projects in wide range of scales from one space to whole neighborhoods.
At Seven Avenue Design we transform the interior of your space to make it functional, personal, according to your space and budget.
Our architectural services include your conceptual and schemati


In [4]:
quant_config = GPTQConfig(
    bits=4,
    group_size=128,
    desc_act=False,
    offload_to_disk=False,
)

model = GPTQModel.load(
    model_id,
    quant_config,
    device="cuda:0",
    dtype=torch.float16,
)

print("model_type：", model.model.config.model_type)

INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-125m/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-125m "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-125m/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-125m/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-125m "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-125m "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-125m/revision/main "HTTP/1.1 200 OK"
Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 1448.56it/s]
INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]
INFO  Loader: loading model directly to CPU (not using meta device or turtle_model)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 17777.06it/s]
INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "eos_token_id": 2,
  "pad_token_id": 1
}

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": 2,
  "pad_token_id": 1
}

INFO  Kernel: loaded -> `[]`                                                    
model_type： opt


## 开始量化

In [5]:
# 清空 PyTorch 缓存分配器里暂时不用的显存，归还给系统
torch.cuda.empty_cache()
# 重置“峰值显存统计”的计数器，重新开始记录
torch.cuda.reset_peak_memory_stats()

start = time.perf_counter()

model.quantize(
    calibration_dataset,
    batch_size=1,
)

elapsed = time.perf_counter() - start

print(f"量化耗时：{elapsed / 60:.2f} 分钟")

print(
    "峰值 allocated：",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB",
)

print(
    "峰值 reserved：",
    round(torch.cuda.max_memory_reserved() / 1024**3, 2),
    "GB",
)

INFO  Packing Kernel: selected: `TorchLinear`                                   
INFO  Packing Kernel: selected: `TorchLinear`                                   
WARN  Calibration dataset size should be more than 256. Current: 16.            
INFO  Quantize: trimmed 2 calibration rows above max_position_embeddings=2048 (longest original length=4291)
INFO  Calibration: Sort in descending order by length                           
INFO  Calibration: Total padded tokens: 0                                       
INFO  Calibration: Total non-padded tokens: 8519                                
INFO  Calibration: Total tokens: 8519                                           
INFO  ModuleLooper: capturing layer inputs from 16 calibration batches          
INFO  +------------+-------+--------+-------+---------+--------+---------+      
INFO  | region     | count | last_s | avg_s | total_s | pct    | source  |      
INFO  +------------+-------+--------+-------+---------+--------+---------+      


## 保存

In [6]:
model.save(str(quant_path))

tokenizer = AutoTokenizer.from_pretrained(
    model_id
)

tokenizer.save_pretrained(str(quant_path))

print("保存完成：", quant_path)

Writing model shards: 0it [00:00, ?it/s]

INFO  Saved Quantize Config: 
{
  "bits": 4,
  "group_size": 128,
  "desc_act": false,
  "lm_head": false,
  "method": "gptq",
  "quant_method": "gptq",
  "format": "gptq",
  "checkpoint_format": "gptq",
  "pack_dtype": "int32",
  "meta": {
    "quantizer": [
      "gptqmodel:7.7.0"
    ],
    "uri": "https://github.com/modelcloud/gptqmodel",
    "damp_percent": 0.05,
    "damp_auto_increment": 0.01,
    "static_groups": false,
    "true_sequential": true,
    "mse": 0.0,
    "gptaq": null,
    "foem": null,
    "act_group_aware": true,
    "fallback": {
      "strategy": "rtn",
      "threshold": "0.5%",
      "smooth": null
    },
    "offload_to_disk": false,
    "telemetry": {
      "device": false
    },
    "offload_to_disk_path": null,
    "pack_impl": "auto",
    "gc_mode": "interval",
    "wait_for_submodule_finalizers": false,
    "auto_forward_data_parallel": true,
    "dense_vram_strategy": "exclusive",
    "dense_vram_strategy_devices": null,
    "moe_vram_strategy": "excl

Files in directory:
quant_log.csv
config.json
generation_config.json
quantize_config.json
Content of saved `generation_config.json`:
{
    "_from_model_config": true,
    "bos_token_id": 2,
    "do_sample": true,
    "eos_token_id": 2,
    "pad_token_id": 1,
    "transformers_version": "5.20.2"
}
Content of saved `config.json`:
{
    "_remove_final_layer_norm": false,
    "activation_dropout": 0.0,
    "activation_function": "relu",
    "architectures": [
        "OPTForCausalLM"
    ],
    "attention_dropout": 0.0,
    "bos_token_id": 2,
    "do_layer_norm_before": true,
    "dropout": 0.1,
    "dtype": "float16",
    "enable_bias": true,
    "eos_token_id": 2,
    "ffn_dim": 3072,
    "hidden_size": 768,
    "init_std": 0.02,
    "layer_norm_elementwise_affine": true,
    "layerdrop": 0.0,
    "max_position_embeddings": 2048,
    "model_type": "opt",
    "num_attention_heads": 12,
    "num_hidden_layers": 12,
    "pad_token_id": 1,
    "prefix": "</s>",
    "quantization_config": {
 

INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-125m "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-125m/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-125m/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/facebook/opt-125m/resolve/27dcfa74d334bc871f3234de431e71c6eeba5dd6/added_tokens.json "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/facebook/opt-125m/resolve/27dcfa74d334bc871f3234de431e71c6eeba5dd6/tokenizer.json "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/facebook/opt-125m/resolve/27dcfa74d334bc871f3234de431e71c6eeba5dd6/chat_template.jinja "HTTP/1.1 404 Not Found"


保存完成： /sigan/ai-ml/models/quantized/opt-125m-gptqmodel-4bit-smoke


## 重新加载并生成

In [8]:
import torch
from gptqmodel import BACKEND, GPTQModel

reloaded_model = GPTQModel.load(
    str(quant_path),
    device="cuda:0",
    dtype=torch.float16,
    backend=BACKEND.GPTQ_TORCH,
)

reloaded_hf_model = reloaded_model.model
reloaded_hf_model.eval()

print("使用后端：", BACKEND.GPTQ_TORCH)

from_quantized: adapter: None
INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  The layer model.decoder.embed_tokens is not quantized.                    
INFO  The layer model.decoder.embed_positions is not quantized.                 
INFO  Kernel: selected: `TorchLinear`                                           
INFO  Kernel: candidates -> `[TorchLinear]`                                     
INFO  Kernel: per-module selections -> `[TorchLinear=72]`                       
INFO  Kernel: selected -> `TorchLinear`.                                        
INFO  Loader: device = cuda:0                                                   
INFO  Loader: honoring explicit device_map request: {'': 'cuda:0'}              
INFO  Loader: device_map = {'': 'cuda:0'}                                       


INFO  Format: Converting `format` from `gptq` to internal `gptq_v2`.            
INFO  Format: Converting GPTQ v1 to v2                                          
INFO  Kernel: selected: `TorchLinear`                                           
INFO  Optimize: `TorchLinear` compilation triggered.                            
INFO  gc.collect() reclaimed 72 objects in 0.138s                               


INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 1,
  "use_cache": true
}

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": 2,
  "pad_token_id": 1
}

INFO  Kernel: loaded -> `[TorchLinear]`                                         
使用后端： BACKEND.GPTQ_TORCH


In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    str(quant_path),
    local_files_only=True,
)

prompt = "Merry Christmas! I'm glad to"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to("cuda:0")

with torch.no_grad():
    output_ids = reloaded_hf_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
    )

print(
    tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
    )
)

In file included from /home/sigan/miniconda3/envs/nowLLM/include/python3.10/Python.h:8,
                 from /home/sigan/miniconda3/envs/nowLLM/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:9:
/home/sigan/miniconda3/envs/nowLLM/include/python3.10/pyconfig.h:1643:9: warning: ‘_POSIX_C_SOURCE’ redefined
 1643 | #define _POSIX_C_SOURCE 200809L
      |         ^~~~~~~~~~~~~~~
In file included from /usr/include/x86_64-linux-gnu/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from /home/sigan/miniconda3/envs/nowLLM/lib/python3.10/site-packages/triton/backends/nvidia/include/cuda.h:56,
                 from /home/sigan/miniconda3/envs/nowLLM/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:1:
/usr/include/features.h:319:10: note: this is the location of the previous definition
  319 | # define _POSIX_C_SOURCE        202405L
      |          ^~~~~~~~~~~~~~~


Merry Christmas! I'm glad to see you're doing well.
Thanks! I'm glad to hear that too!


# 量化 OPT-6.7B

**重启内核，释放刚才的小模型**

## 逐分片转换成本地 .safetensors

In [2]:
from pathlib import Path
from huggingface_hub import snapshot_download

source_model_dir = Path(
    "/sigan/ai-ml/models/dense/opt-6.7b"
)

source_model_dir.mkdir(
    parents=True,
    exist_ok=True,
)

downloaded_path = snapshot_download(
    repo_id="facebook/opt-6.7b",
    local_dir=str(source_model_dir),
    cache_dir="/sigan/ai-ml/huggingface/hub",
    allow_patterns=[
        "config.json",
        "generation_config.json",
        "pytorch_model*.bin",
        "pytorch_model.bin.index.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "vocab.json",
        "merges.txt",
        "LICENSE.md",
        "README.md",
    ],
    max_workers=4,
)

print("模型目录：", downloaded_path)

/home/sigan/miniconda3/envs/nowLLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 11 files: 100%|██████████| 11/11 [06:57<00:00, 37.96s/it]

模型目录： /mnt/e/sigan/ai-ml/models/dense/opt-6.7b


In [3]:
# 检查文件
for path in sorted(source_model_dir.iterdir()):
    if path.is_file():
        print(
            path.name,
            round(path.stat().st_size / 1024**3, 3),
            "GB",
        )

.gitattributes 0.0 GB
LICENSE.md 0.0 GB
README.md 0.0 GB
config.json 0.0 GB
flax_model-00001-of-00002.msgpack 9.308 GB
flax_model-00002-of-00002.msgpack 3.095 GB
flax_model.msgpack.index.json 0.0 GB
generation_config.json 0.0 GB
merges.txt 0.0 GB
pytorch_model-00001-of-00002.bin 9.277 GB
pytorch_model-00002-of-00002.bin 3.126 GB
pytorch_model.bin.index.json 0.0 GB
special_tokens_map.json 0.0 GB
tf_model-00001-of-00002.h5 9.277 GB
tf_model-00002-of-00002.h5 3.126 GB
tf_model.h5.index.json 0.0 GB
tokenizer_config.json 0.0 GB
vocab.json 0.001 GB


### 逐分片转换为 safetensors

In [12]:
import gc
import json
import os
import shutil
import time
from collections import defaultdict
from pathlib import Path

import torch
from safetensors import safe_open
from safetensors.torch import save_file

In [13]:
source_model_dir = Path(
    "/sigan/ai-ml/models/dense/opt-6.7b"
)

safetensors_model_dir = Path(
    "/sigan/ai-ml/models/dense/opt-6.7b-safetensors"
)

safetensors_model_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print("源模型：", source_model_dir)
print("safetensors 输出：", safetensors_model_dir)

源模型： /sigan/ai-ml/models/dense/opt-6.7b
safetensors 输出： /sigan/ai-ml/models/dense/opt-6.7b-safetensors


In [14]:
# 读取原始权重索引
index_path = (
    source_model_dir
    / "pytorch_model.bin.index.json"
)

if not index_path.is_file():
    raise FileNotFoundError(
        f"缺少权重索引：{index_path}"
    )

with index_path.open(
    "r",
    encoding="utf-8",
) as file:
    bin_index = json.load(file)

weight_map = bin_index["weight_map"]
bin_shards = sorted(set(weight_map.values()))

print("原始分片数量：", len(bin_shards))
print("原始权重数量：", len(weight_map))

for shard_name in bin_shards:
    print(shard_name)

原始分片数量： 2
原始权重数量： 516
pytorch_model-00001-of-00002.bin
pytorch_model-00002-of-00002.bin


In [15]:
# 复制配置和 tokenizer 文件

auxiliary_files = [
    "config.json",
    "generation_config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "vocab.json",
    "merges.txt",
    "LICENSE.md",
    "README.md",
]

for filename in auxiliary_files:
    source = source_model_dir / filename

    if source.is_file():
        destination = safetensors_model_dir / filename
        shutil.copy2(source, destination)
        print("已复制：", filename)

已复制： config.json
已复制： generation_config.json
已复制： tokenizer_config.json
已复制： special_tokens_map.json
已复制： vocab.json
已复制： merges.txt
已复制： LICENSE.md
已复制： README.md


In [16]:
# 定义单分片转换函数
def convert_one_shard(bin_filename):
    bin_path = source_model_dir / bin_filename

    safe_filename = (
        bin_filename
        .replace("pytorch_model-", "model-")
        .replace(".bin", ".safetensors")
    )

    output_path = safetensors_model_dir / safe_filename
    temporary_path = Path(str(output_path) + ".partial")

    if output_path.is_file():
        print("已存在，跳过：", output_path.name)
        return bin_filename, safe_filename

    if temporary_path.exists():
        temporary_path.unlink()

    print("=" * 70)
    print("开始转换：", bin_filename)
    print(
        "文件大小 GB：",
        bin_path.stat().st_size / 1024**3,
    )

    start = time.perf_counter()

    state_dict = torch.load(
        bin_path,
        map_location="cpu",
        mmap=True,
        weights_only=True,
    )

    tensors = {}

    for key, tensor in state_dict.items():
        tensor = tensor.detach().cpu()

        if not tensor.is_contiguous():
            tensor = tensor.contiguous()

        tensors[key] = tensor

    # 找出共享同一块底层存储的张量
    storage_owners = defaultdict(list)

    for name, tensor in tensors.items():
        storage_key = tensor.untyped_storage().data_ptr()
        storage_owners[storage_key].append(name)

    shared_count = 0

    for names in storage_owners.values():
        if len(names) <= 1:
            continue

        # 保留第一个，其余的复制成独立存储
        for name in names[1:]:
            tensors[name] = tensors[name].clone()
            shared_count += 1

    print("处理共享存储张量数量：", shared_count)

    # 字典使用 save_file，而不是 save_model
    save_file(
        tensors,
        str(temporary_path),
        metadata={"format": "pt"},
    )

    os.replace(
        temporary_path,
        output_path,
    )

    elapsed = time.perf_counter() - start

    print(
        "转换完成：",
        output_path.name,
        f"耗时 {elapsed / 60:.2f} 分钟",
    )

    del tensors
    del state_dict
    gc.collect()

    return bin_filename, safe_filename

In [17]:
shard_name_mapping = {}

for bin_filename in bin_shards:
    old_name, new_name = convert_one_shard(
        bin_filename
    )
    shard_name_mapping[old_name] = new_name

print("分片转换完成：")
print(shard_name_mapping)

开始转换： pytorch_model-00001-of-00002.bin
文件大小 GB： 9.276672226376832
处理共享存储张量数量： 0
转换完成： model-00001-of-00002.safetensors 耗时 16.04 分钟
开始转换： pytorch_model-00002-of-00002.bin
文件大小 GB： 3.1258540088310838
处理共享存储张量数量： 0
转换完成： model-00002-of-00002.safetensors 耗时 6.42 分钟
分片转换完成：
{'pytorch_model-00001-of-00002.bin': 'model-00001-of-00002.safetensors', 'pytorch_model-00002-of-00002.bin': 'model-00002-of-00002.safetensors'}


In [18]:
# 生成 safetensors 索引
safe_weight_map = {
    tensor_name: shard_name_mapping[old_shard_name]
    for tensor_name, old_shard_name in weight_map.items()
}

safe_index = {
    "metadata": {
        "total_size": bin_index.get(
            "metadata",
            {},
        ).get(
            "total_size",
            0,
        ),
    },
    "weight_map": safe_weight_map,
}

safe_index_path = (
    safetensors_model_dir
    / "model.safetensors.index.json"
)

with safe_index_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        safe_index,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("已写入：", safe_index_path)

已写入： /sigan/ai-ml/models/dense/opt-6.7b-safetensors/model.safetensors.index.json


### 验证转换结果

In [19]:
safetensors_files = sorted(
    safetensors_model_dir.glob("*.safetensors")
)

print("分片数量：", len(safetensors_files))

total_size = sum(
    path.stat().st_size
    for path in safetensors_files
) / 1024**3

print("总大小 GB：", total_size)

for path in safetensors_files:
    print(
        path.name,
        round(
            path.stat().st_size / 1024**3,
            3,
        ),
        "GB",
    )

分片数量： 2
总大小 GB： 12.40242812782526
model-00001-of-00002.safetensors 9.277 GB
model-00002-of-00002.safetensors 3.126 GB


In [22]:
# 逐个打开，确认文件可以读取
for path in safetensors_files:
    with safe_open(
        str(path),
        framework="pt",
        device="cpu",
    ) as file:
        keys = list(file.keys())

    print(
        path.name,
        "tensor 数量：",
        len(keys),
    )

model-00001-of-00002.safetensors tensor 数量： 384
model-00002-of-00002.safetensors tensor 数量： 132


In [23]:
# 检查必要文件
required_files = [
    "config.json",
    "model.safetensors.index.json",
]

for filename in required_files:
    path = safetensors_model_dir / filename
    assert path.is_file(), f"缺少文件：{path}"

print("基础校验通过")

基础校验通过


## 重启内核开始量化

In [1]:
import gc
import os
import time
from pathlib import Path

os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True",
)

import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from gptqmodel import GPTQConfig, GPTQModel

/home/sigan/miniconda3/envs/nowLLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


W1101 20:26:09.914000 32963 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1101 20:26:10.032000 32963 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.7.0
Transformers : 5.20.2
Torch        : 2.14.1+cu130
Triton       : 3.8.0


In [2]:
model_id = (
    "/sigan/ai-ml/models/dense/"
    "opt-6.7b-safetensors"
)

quant_path = Path(
    "/sigan/ai-ml/models/quantized/"
    "opt-6.7b-gptqmodel-4bit-local"
)

offload_path = Path(
    "/sigan/ai-ml/work/gptqmodel-offload/"
    "opt-6.7b-local"
)

quant_path.mkdir(
    parents=True,
    exist_ok=True,
)

offload_path.mkdir(
    parents=True,
    exist_ok=True,
)

print("源模型：", model_id)
print("量化输出：", quant_path)
print("offload 路径：", offload_path)

源模型： /sigan/ai-ml/models/dense/opt-6.7b-safetensors
量化输出： /sigan/ai-ml/models/quantized/opt-6.7b-gptqmodel-4bit-local
offload 路径： /sigan/ai-ml/work/gptqmodel-offload/opt-6.7b-local


### 准备校准数据
**第一轮建议使用 128 条，而不是直接上 1024 条**

In [3]:
calibration_dataset = load_dataset(
    "allenai/c4",
    data_files=(
        "en/c4-train.00001-of-01024.json.gz"
    ),
    split="train",
).select(range(128))

print(
    "校准样本数：",
    len(calibration_dataset),
)

INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/allenai/c4/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/api/resolve-cache/datasets/allenai/c4/1588ec454efa1a09f29cd18ddd04fe05fc8653a2/README.md?%2Fdatasets%2Fallenai%2Fc4%2Fresolve%2Fmain%2FREADME.md=&etag=%222a750bf152e98e6eb2a14a6c2bd58c940c3cf504%22 "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/allenai/c4/resolve/1588ec454efa1a09f29cd18ddd04fe05fc8653a2/c4.py "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/allenai/c4/allenai/c4.py "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/datasets/allenai/c4/revision/1588ec454efa1a09f29cd18ddd04fe05fc8653a2 "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/allenai/c4/resolve/1588ec454efa1a09f29cd18ddd04fe05fc8653a2/.huggingface.yaml "HTTP/1.1 404 No

校准样本数： 128


### 加载 OPT-6.7B

In [4]:
quant_config = GPTQConfig(
    bits=4,
    group_size=128,
    desc_act=False,
    offload_to_disk=True,
    offload_to_disk_path=str(offload_path),
)

### 开始量化

In [5]:
model = GPTQModel.load(
    model_id,
    quant_config,
    device="cuda:0",
    dtype=torch.float16,
)

print(
    "model_type：",
    model.model.config.model_type,
)

INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]
INFO  Loader: using checkpoint-backed lazy turtle source for `/sigan/ai-ml/models/dense/opt-6.7b-safetensors`


INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 1,
  "use_cache": true
}

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": 2,
  "pad_token_id": 1
}

INFO  Kernel: loaded -> `[]`                                                    
model_type： opt


In [6]:
# PyTorch 会记录从程序开始（或上次重置）以来，GPU 显存使用的峰值。可以用 torch.cuda.max_memory_allocated() 查询这个值
torch.cuda.reset_peak_memory_stats()

free_bytes, total_bytes = torch.cuda.mem_get_info()

print(
    "量化前空闲显存 GB：",
    free_bytes / 1024**3,
)

start = time.perf_counter()

model.quantize(
    calibration_dataset,
    batch_size=1,
)

elapsed = time.perf_counter() - start

print(
    f"量化耗时：{elapsed / 60:.2f} 分钟"
)

print(
    "峰值 allocated GB：",
    torch.cuda.max_memory_allocated() / 1024**3,
)

print(
    "峰值 reserved GB：",
    torch.cuda.max_memory_reserved() / 1024**3,
)

量化前空闲显存 GB： 14.56640625
INFO  Packing Kernel: selected: `TorchLinear`                                   
INFO  Packing Kernel: selected: `TorchLinear`                                   
WARN  Calibration dataset size should be more than 256. Current: 128.           
INFO  Quantize: trimmed 3 calibration rows above max_position_embeddings=2048 (longest original length=6876)
INFO  Calibration: Sort in descending order by length                           
INFO  Calibration: Total padded tokens: 0                                       
INFO  Calibration: Total non-padded tokens: 51839                               
INFO  Calibration: Total tokens: 51839                                          
INFO  Disk subsystem write throughput detected at 1764.9 MB/s.                  
INFO  ModuleLooper: capturing layer inputs from 128 calibration batches         
INFO  Offloading base modules to disk...                                        
INFO  +------------+-------+--------+-------+---------+--

INFO  GPTQ block update: compiling torch.ops JIT extension in `/home/sigan/.cache/gptqmodel/torch_extensions/gptq_block/5f294f4504298a10`.
INFO  GPTQ block update: torch.ops JIT compilation failed in 0.0s; using fallback path.
INFO  +------+-----+-----+-------+------+---+------+---+---+------+-----+------+
INFO  | p... | ... | ... | fe... | d... | l | s... | d | t | f... | ... | d... |
INFO  +------+-----+-----+-------+------+---+------+---+---+------+-----+------+
INFO  | ... | 0 | ... | 40... | f16:... | ... | 51 | ... | ... | 0. | ... |    |
INFO  +-----+---+-----+-------+---------+-----+----+-----+-----+----+-----+----+
INFO  | ... | 0 | ... | 40... | f16:... | ... | 51 | ... | ... | 0. | ... |    |
INFO  +-----+---+-----+-------+---------+-----+----+-----+-----+----+-----+----+
INFO  | ... | 0 | ... | 40... | f16:... | ... | 51 | ... | ... | 0. | ... |    |
INFO  +-----+---+-----+-------+---------+-----+----+-----+-----+----+-----+----+
INFO  | ... | 0 | ... | 40... | f16:... | ..

### 保存模型

In [7]:
model.save(str(quant_path))

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    local_files_only=True,
)

tokenizer.save_pretrained(
    str(quant_path)
)

print("量化模型已保存：", quant_path)

Writing model shards: 0it [00:00, ?it/s]

INFO  Saved Quantize Config: 
{
  "bits": 4,
  "group_size": 128,
  "desc_act": false,
  "lm_head": false,
  "method": "gptq",
  "quant_method": "gptq",
  "format": "gptq",
  "checkpoint_format": "gptq",
  "pack_dtype": "int32",
  "meta": {
    "quantizer": [
      "gptqmodel:7.7.0"
    ],
    "uri": "https://github.com/modelcloud/gptqmodel",
    "damp_percent": 0.05,
    "damp_auto_increment": 0.01,
    "static_groups": false,
    "true_sequential": true,
    "mse": 0.0,
    "gptaq": null,
    "foem": null,
    "act_group_aware": true,
    "fallback": {
      "strategy": "rtn",
      "threshold": "0.5%",
      "smooth": null
    },
    "offload_to_disk": true,
    "telemetry": {
      "device": false
    },
    "offload_to_disk_path": "/sigan/ai-ml/work/gptqmodel-offload/opt-6.7b-local",
    "pack_impl": "auto",
    "gc_mode": "interval",
    "wait_for_submodule_finalizers": false,
    "auto_forward_data_parallel": true,
    "dense_vram_strategy": "exclusive",
    "dense_vram_strategy

Files in directory:
quant_log.csv
config.json
generation_config.json
quantize_config.json
Content of saved `generation_config.json`:
{
    "_from_model_config": true,
    "bos_token_id": 2,
    "do_sample": true,
    "eos_token_id": 2,
    "pad_token_id": 1,
    "transformers_version": "5.20.2"
}
Content of saved `config.json`:
{
    "_remove_final_layer_norm": false,
    "activation_dropout": 0.0,
    "activation_function": "relu",
    "architectures": [
        "OPTForCausalLM"
    ],
    "attention_dropout": 0.0,
    "bos_token_id": 2,
    "do_layer_norm_before": true,
    "dropout": 0.1,
    "dtype": "float16",
    "enable_bias": true,
    "eos_token_id": 2,
    "ffn_dim": 16384,
    "hidden_size": 4096,
    "init_std": 0.02,
    "layer_norm_elementwise_affine": true,
    "layerdrop": 0.0,
    "max_position_embeddings": 2048,
    "model_type": "opt",
    "num_attention_heads": 32,
    "num_hidden_layers": 32,
    "pad_token_id": 1,
    "prefix": "</s>",
    "quantization_config": {

# 加载 OPT-6.7B GPTQ 并生成

In [4]:
from gptqmodel import BACKEND, GPTQModel
import torch
from transformers import AutoTokenizer
from gptqmodel import BACKEND, GPTQModel
from pathlib import Path

quant_path = Path(
    "/sigan/ai-ml/models/quantized/"
    "opt-6.7b-gptqmodel-4bit-local"
)

tokenizer = AutoTokenizer.from_pretrained(
    str(quant_path),
    local_files_only=True,
)

# GPTQModel 负责量化相关的逻辑：加载 4-bit 权重、管理 backend、处理量化配置
reloaded_model = GPTQModel.load(
    str(quant_path),
    device="cuda:0",
    dtype=torch.float16,
    backend=BACKEND.GPTQ_TORCH,
)

# 底层 .model 才是标准的 OPTForCausalLM 这类结构，可以像普通 HuggingFace 模型一样调用
reloaded_hf_model = reloaded_model.model
# 把模型切换到推理模式（evaluation mode）
reloaded_hf_model.eval()

print("推理后端：", BACKEND.GPTQ_TORCH)

from_quantized: adapter: None
INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  The layer model.decoder.embed_tokens is not quantized.                    
INFO  The layer model.decoder.embed_positions is not quantized.                 
INFO  Kernel: selected: `TorchLinear`                                           
INFO  Kernel: candidates -> `[TorchLinear]`                                     
INFO  Kernel: per-module selections -> `[TorchLinear=192]`                      
INFO  Kernel: selected -> `TorchLinear`.                                        
INFO  Loader: device = cuda:0                                                   
INFO  Loader: honoring explicit device_map request: {'': 'cuda:0'}              
INFO  Loader: device_map = {'': 'cuda:0'}                                       


INFO  Format: Converting `format` from `gptq` to internal `gptq_v2`.            
INFO  Format: Converting GPTQ v1 to v2                                          
INFO  Kernel: selected: `TorchLinear`                                           
INFO  Optimize: `TorchLinear` compilation triggered.                            
INFO  gc.collect() reclaimed 144 objects in 0.117s                              


INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 1,
  "use_cache": true
}

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": 2,
  "pad_token_id": 1
}

INFO  Kernel: loaded -> `[TorchLinear]`                                         
推理后端： BACKEND.GPTQ_TORCH


## 生成测试

In [5]:
prompt = "Merry Christmas! I'm glad to"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to("cuda:0")

with torch.no_grad():
    output_ids = reloaded_hf_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
    )

print(
    tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
    )
)

Merry Christmas! I'm glad to see you're still around.
I'm still around, just not as much as I used to be.


## 检查量化层

In [7]:
q_proj = (
    reloaded_hf_model.model
    .decoder.layers[0]
    .self_attn.q_proj
)

# 使用正确的属性名
print("输入维度：", q_proj.in_features)
print("输出维度：", q_proj.out_features)
print("bits：", q_proj.bits)
print("group_size：", q_proj.group_size)

print(
    "qweight：",
    q_proj.qweight.shape,
    q_proj.qweight.dtype,
)

print(
    "scales：",
    q_proj.scales.shape,
    q_proj.scales.dtype,
)

输入维度： 4096
输出维度： 4096
bits： 4
group_size： 128
qweight： torch.Size([512, 4096]) torch.int32
scales： torch.Size([32, 4096]) torch.float16


## 检查大小

In [8]:
def directory_size_gb(path):
    return sum(
        file.stat().st_size
        for file in Path(path).rglob("*")
        if file.is_file()
    ) / 1024**3

print(
    "量化目录 GB：",
    directory_size_gb(quant_path),
)

print(
    "模型张量占用 GB：",
    reloaded_hf_model.get_memory_footprint() / 1024**3,
)

量化目录 GB： 3.5274024195969105
模型张量占用 GB： 3.523967742919922
